# handoff_lean85 — lean-85 최종 운영 노트북

**확정**: lean-85(85피처) + XGBoost 동결 파라미터 + **정기 재학습** · 제작 2026-07-19 · 패키지 v1.0
**전달 규약**: 미러 미검증본(v14 전달 방식) — 정적 검토만 거침. **첫 실행 오류 시 메시지 주면 즉시 수정.**

| 항목 | 내용 |
|---|---|
| 모델 | lean-85 = Conservative-GA(99) − 시간불안정 14 (v13 REPORT_12) · XGB(M5 튜닝, 246r, seed42) |
| 현업 인용 수치 | **시간축 pooled RMSE 99.84 · honest R² 0.8545** (주간 재학습 전제, B2′=v14 V0 재현) |
| same-era 수치 | lot-CV stable 66.83 / seed_mean 66.956 — "같은 시기 새 Lot" 전용, **현업 인용 금지** |
| 재학습 정책 | ① 주간 7일 정기 ② 요란 PM 기입 즉시 ③ PSI 경보·주간 RMSE 급등 수시 |
| 신뢰 플래그 | 요란 PM 후 7일(레짐-온셋) = `low_confidence=1` — B2′ 실증상 오차 급증(147/113/109) |

**전제 파일**: `frozen/lean85_features.json` · `frozen/tuned_params_xgb.json` · `../문제1(하)/train_data.csv` · 루트 `pm_log.json` · (동결 대조용) `modeling_v13/colab_GA/core10_meta_wf.csv` · `modeling_v13/data/v13_fdc_pool_wf_oof.csv.gz`

**실행 순서**: 셀1 환경 → 셀2 피처 재현+동결 대조(A1/A2/A3) → 셀3 최종 학습·버전 저장 → 셀4 신규 예측 데모 → 셀5 재학습 트리거·드리프트 → 셀6(옵션) B2′ 수용검사

**게이트**: A1 메타 8종 = `core10_meta_wf.csv` 완전 일치 · A2 센서 표본 10컬럼 = v13 풀 일치(6유효숫자 반올림 허용) · A3 floor(R10)·누수(R2) assert

**규율**: R2 누수 금지 · **R3 valid/test 정답 미접근 — 이 노트북은 채점하지 않는다** · R6 공식 판정=사용자 로컬 venv · R10 필수 5센서 floor

In [ ]:
# [셀1] 환경 + 동결 스펙 로드
import sys, json, platform
from pathlib import Path
import numpy as np, pandas as pd, xgboost as xgb

PKG = Path.cwd() if (Path.cwd() / "lean85_pipeline.py").exists() else Path.cwd() / "handoff_lean85"
assert (PKG / "lean85_pipeline.py").exists(), "handoff_lean85 폴더에서 실행하세요"
sys.path.insert(0, str(PKG))
import lean85_pipeline as lp

LEAN, PARAMS, ROUNDS = lp.load_frozen()
TRAIN = lp.find_file("train_data.csv")
PMLOG = lp.find_file("pm_log.json")
print(f"python {platform.python_version()} | xgboost {xgb.__version__} | pandas {pd.__version__}")
print(f"lean-85 n={len(LEAN)} | floor={lp.floor_ok(LEAN)[1]} | XGB rounds={ROUNDS}")
print(f"train  = {TRAIN}")
print(f"pm_log = {PMLOG} -> {lp.parse_pm_log(PMLOG)}")

In [ ]:
# [셀2] 피처 재현 + 동결 대조 (A1 메타 / A2 센서 / A3 규율)
raw = pd.read_csv(TRAIN)
tbl = lp.build_wafer_table(raw, PMLOG, LEAN)
print(f"웨이퍼 테이블 {tbl.shape} | 기간 {str(tbl['wf_ts'].min())[:10]} ~ {str(tbl['wf_ts'].max())[:10]}")

# A1: core 메타 8종 — v13 동결본(core10_meta_wf.csv)과 완전 일치해야 함
fm = pd.read_csv(lp.find_file("core10_meta_wf.csv"))
mg = tbl.merge(fm, on="C64", suffixes=("", "_frz"))
assert len(mg) == len(fm) == len(tbl), f"웨이퍼 수 불일치: tbl {len(tbl)} vs frozen {len(fm)}"
for c in lp.META_COLS:
    assert np.allclose(mg[c], mg[f"{c}_frz"], rtol=1e-9, atol=1e-9), f"A1 실패: {c}"
print(f"A1 PASS — 메타 {len(lp.META_COLS)}종 x {len(mg):,} 웨이퍼, 동결본과 완전 일치")

# A2: 센서 집계 표본 10컬럼 — v13 풀과 일치 (풀 CSV는 6유효숫자 반올림 -> rtol 2e-5)
SAMPLE = ["C17_max_step4", "C11_min_step4", "C31_mean_step4", "C15_max_step1", "C16_max_step1",
          "C25_std_step1", "C58_mean_step4", "C62_mean_step1", "C4_std_step6", "C60_std_step4"]
pool = pd.read_csv(lp.find_file("v13_fdc_pool_wf_oof.csv.gz"), usecols=["C64"] + SAMPLE)
pg = tbl.merge(pool, on="C64", suffixes=("", "_frz"))
assert len(pg) == len(tbl)
for c in SAMPLE:
    a, b = pg[c].to_numpy(float), pg[f"{c}_frz"].to_numpy(float)
    assert np.array_equal(np.isnan(a), np.isnan(b)), f"A2 실패(NaN 패턴): {c}"
    assert np.allclose(a[~np.isnan(a)], b[~np.isnan(b)], rtol=2e-5), f"A2 실패(값): {c}"
print(f"A2 PASS — 센서 표본 {len(SAMPLE)}컬럼, 동결 풀과 일치 (NaN 패턴 포함)")
print("A3 PASS — floor(R10)·누수(R2) assert는 build_wafer_table/load_frozen 내부에서 통과")

In [ ]:
# [셀3] 최종 학습 (전체 train) + 버전 저장 — 운영 재학습과 동일 경로(retrain)
vdir, model, mani = lp.retrain(raw, PMLOG, out_dir="models", tag="v1")
print(f"저장: {vdir}")
show = {k: mani[k] for k in ["package", "n_wafers_train", "train_period",
                             "features_sha1", "n_estimators", "env"]}
print(json.dumps(show, ensure_ascii=False, indent=2))
tr_rmse, _ = lp.evaluate_rmse(model, tbl, LEAN)
print(f"(참고) train-fit RMSE {tr_rmse:.2f} — 적합 확인용일 뿐 성능 지표 아님. 현업 인용은 시간축 99.84 (REPORT_01 §5)")

In [ ]:
# [셀4] 신규 데이터 예측 데모 — valid_X (R3: 정답 미접근·채점 없음)
try:
    rawv = pd.read_csv(lp.find_file("valid_X.csv"))
    tv = lp.build_wafer_table(rawv, PMLOG, LEAN)
    preds = lp.predict_lean85(model, tv, LEAN)
    out_p = vdir / "preds_valid_demo.csv"
    preds.to_csv(out_p, index=False)
    n_low = int(preds["low_confidence"].sum())
    print(f"valid_X {len(preds):,} 웨이퍼 예측 -> {out_p}")
    print(f"low_confidence(레짐-온셋) {n_low:,} ({n_low / max(len(preds), 1):.1%})")
    print(preds.head(5).to_string(index=False))
except FileNotFoundError as e:
    print("valid_X.csv 없음 — 데모 생략:", e)

In [ ]:
# [셀5] 재학습 SOP — 트리거 판정 + 입력 드리프트(PSI) 모니터링
ok, reasons = lp.should_retrain(mani, PMLOG)
print("재학습 트리거:", "필요" if ok else "불필요(대기)")
for r in reasons:
    print(" -", r)

# PSI 데모: 학습창 전반 vs 최근 1주 (운영: ref=직전 학습 테이블, new=신규 주간 테이블)
cut = tbl["wf_ts"].max() - pd.Timedelta(days=7)
rep = lp.drift_report(tbl[tbl["wf_ts"] <= cut], tbl[tbl["wf_ts"] > cut], LEAN, top=10)
print(rep.to_string(index=False))

print("""
운영 절차 (요약 — 상세는 handoff_lean85_README.md §재학습 SOP):
 1) 트리거: 주간 7일 정기 / 요란 PM 기입 즉시 / PSI 경보·주간 RMSE 급등 수시
 2) 실행:   python retrain_lean85.py --data <누적 raw> --pm-log <최신 pm_log> --tag weekly
 3) 검증:   (환경/코드 변경 시) --acceptance 로 B2' 재현 99.840±0.5 확인
 4) 교체:   predict 의 --model 을 신규 버전 폴더로 변경 (구버전 보존 = 롤백 대비)""")

In [ ]:
# [셀6·옵션] B2' 롤링-재학습 수용검사 — RUN_ACCEPT=True 로 바꿔 실행 (수 분 소요)
RUN_ACCEPT = False   # 최초 인수 시 · 환경/코드 변경 시 True 권장
if RUN_ACCEPT:
    acc = lp.walkforward_acceptance(tbl, LEAN, PARAMS, ROUNDS)
    mani["acceptance"] = acc
    json.dump(mani, open(vdir / "manifest.json", "w", encoding="utf-8"),
              ensure_ascii=False, indent=2)
else:
    print("생략(RUN_ACCEPT=False). 기대: pooled 99.840 ± 0.5 — fold별 기준표는 REPORT_01 §4")

## 마무리 — 인용 규칙·체크리스트

**수치 인용 규칙 (필수)**
- 현업/보고 인용: **시간축 pooled RMSE 99.84 · honest R² 0.8545** (주간 재학습 전제)
- lot-CV 66.83~66.96 은 "같은 시기 새 Lot"(후향·대회축) 전용 — 미래 성능 근거로 인용 금지
- 무재학습 254.9 = 배포 불가 실증 → **재학습은 선택이 아니라 전제**

**운영 체크리스트**
- [ ] 주간(또는 요란 PM 직후) `retrain_lean85.py` 실행, 버전 폴더 확인
- [ ] 예측 산출물의 `low_confidence` 웨이퍼는 참고용 처리 (레짐-온셋)
- [ ] pm_log.json 은 요란 PM만, 기입 즉시 재학습 (조용 PM은 미기입 = C33이 담당)
- [ ] 월 1회 `drift_report` PSI 점검 · 주간 RMSE 추이 기록 (README 실험 로그에 누적)

산출물: `models/lean85_<stamp>_v1/` (모델+manifest) · 문서: `handoff_lean85_README.md` · `REPORT/handoff_lean85_REPORT_01_lean85_확정.md`